# Attention Mechanisms: Deep Dive
### Interactive Notebook for AI/ML Interview Preparation

📺 **Video Lecture:** [https://youtu.be/te7D9Al7mpw](https://youtu.be/te7D9Al7mpw)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
print('Libraries loaded!')

---
## 1. Additive (Bahdanau) vs Multiplicative (Luong) Attention

In [ ]:
def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

seq_len, d = 6, 8
query = np.random.randn(d)  # decoder hidden state
keys = np.random.randn(seq_len, d)  # encoder outputs
values = keys.copy()

# Luong (multiplicative): score = q^T * K
luong_scores = keys @ query
luong_weights = softmax(luong_scores)

# Bahdanau (additive): score = v^T * tanh(W1*q + W2*k)
W1 = np.random.randn(d, d) * 0.1
W2 = np.random.randn(d, d) * 0.1
v = np.random.randn(d) * 0.1
bahdanau_scores = np.array([v @ np.tanh(W1 @ query + W2 @ k) for k in keys])
bahdanau_weights = softmax(bahdanau_scores)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(seq_len)
ax.bar(x - 0.15, luong_weights, 0.3, label='Luong (multiplicative)', alpha=0.7)
ax.bar(x + 0.15, bahdanau_weights, 0.3, label='Bahdanau (additive)', alpha=0.7)
ax.set_xlabel('Key position'); ax.set_ylabel('Attention weight')
ax.set_title('Attention Score Comparison'); ax.legend()
plt.tight_layout(); plt.show()

---
## 2. KV-Cache for Inference

In [ ]:
# Simulate autoregressive generation with and without KV-cache
d_model = 64
max_len = 50

# Without cache: recompute all K,V at each step
ops_no_cache = [t * d_model for t in range(1, max_len + 1)]  # O(t*d) per step
total_no_cache = sum(ops_no_cache)

# With cache: only compute new K,V, reuse previous
ops_with_cache = [d_model for _ in range(max_len)]  # O(d) per step
total_with_cache = sum(ops_with_cache)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, max_len+1), np.cumsum(ops_no_cache), 'r-', label='Without KV-cache')
ax.plot(range(1, max_len+1), np.cumsum(ops_with_cache), 'b-', label='With KV-cache')
ax.set_xlabel('Generation step'); ax.set_ylabel('Cumulative operations')
ax.set_title('KV-Cache Saves Computation During Generation')
ax.legend(); plt.tight_layout(); plt.show()
print(f'Speedup at step {max_len}: {total_no_cache/total_with_cache:.1f}x')

---
## Key Interview Takeaways

1. **Additive attention** — more flexible, works for mismatched dimensions
2. **Multiplicative attention** — simpler, faster, used in Transformers
3. **KV-cache** — stores previous keys/values; essential for fast autoregressive inference
4. **Multi-head** — parallel heads capture diverse patterns
5. **Causal masking** — prevents decoder from seeing future

---

<small><em>© 2026 AI Nirvana · Disclaimer: Provided as is. No liability assumed.</em></small>